# Training model with distributed learning pipeline (experimental)


## Verification of dependencies and their versions

In [ ]:
import time
from datetime import datetime

import kubeflow
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print("All imports successful!")

## PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes. `TrainerClient()` verifies that you have required access to the Kubernetes cluster. Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [ ]:

from kubeflow.trainer import CustomTrainer, TrainerClient
config = kubeflow.trainer.KubernetesBackendConfig()

client = TrainerClient()
trainer = TrainerClient()
# trainer = TrainerClient(backend_config=config) #TOTEST

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob. Additionally, it might show available accelerator type and number of available resources.

In [ ]:
for runtime in trainer.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [ ]:
from kubeflow.trainer import TrainerClient, CustomTrainer, options
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer


# ------------------------------------------------
# Configuration of resources
# ------------------------------------------------

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 1

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "5",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

# Set github container registry
GITHUB_CONTAINER_REGISTRY = "ghcr.io/xfetus/fetal-ultrasound-edm2/fetal-ultrasound-edm2-distributed-learning:v0.0.4"

# Set up arguments of your function
FUNC_ARGS={
    "checkpoint_path": "/scratch-volume/YOUR_CHECKPOINT_PATH", # pragma: allowlist secret
    "save_interval": 10,
    "resume": False,
    "epochs": 50,
    "batch_size": 128,
}

# ------------------------------------------------
# Seeting up to mount checkpoint volume
# ------------------------------------------------
volume_name = "scratch-volume"
pvc_name = "scratch-volume"
mount_scrath_path = "/scratch-volume"

pod_volumes = [
    {
        "name": volume_name,
        "persistentVolumeClaim": {
            "claimName": pvc_name
        }
    }
]
volume_mounts = [
    {
        "name": volume_name,
        "mountPath": mount_scrath_path
    }
]
container_override = options.ContainerOverride(
    name="node",
    volume_mounts=volume_mounts
)
pod_spec_override = options.PodSpecOverride(
    volumes=pod_volumes,
    containers=[container_override]
)
pod_template_override = options.PodTemplateOverride(
    target_jobs=["node"],
    spec=pod_spec_override
)
pod_template_overrides = options.PodTemplateOverrides(
    pod_template_override
)


ENV_VARS = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0"
}

# command = TrainerCommand(
#     command=[
#         "torchrun",
#         "--standalone",
#         "--nproc_per_node=1",
#         "train_edm2.py",
#         "--outdir", "~/scratch-volume/FETAL_PLANES_ZENODO/OUTPUT_DIRECTORY", # pragma: allowlist secret
#         "--data", "~/scratch-volume/FETAL_PLANES_ZENODO", # pragma: allowlist secret
#         "--batch", "8",
#         "--preset", "edm2-img512-s",
#         "--batch-gpu", "8",
#     ]
# )


command = TrainerCommand(
    command=[
        "/opt/conda/bin/python3.11",
        "-m",
        "torch.distributed.run",
        "--nproc_per_node=1",
        "train_edm2.py",
        "--outdir", "/scratch-volume/FETAL_PLANES_ZENODO/OUTPUT_DIRECTORY", # pragma: allowlist secret
        "--data", "/scratch-volume/FETAL_PLANES_ZENODO", # pragma: allowlist secret
        "--batch", "8",
        "--preset", "edm2-img512-s",
        "--batch-gpu", "8",
    ]
)

In [ ]:
job_id = trainer.train(
    runtime=torch_runtime,
    trainer=CustomTrainerContainer(
        image=GITHUB_CONTAINER_REGISTRY,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
        env=ENV_VARS        
    ),
    options=[command, pod_template_overrides],
)


In [ ]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")

In [ ]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


# Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [ ]:
client.delete_job(job_id)